In [1]:
try :
    from datetime import date
    import datetime
    import time
    from catboost import Pool
    from catboost import CatBoostClassifier
    from elasticsearch import Elasticsearch
    from elasticsearch import helpers
    import os
    import sys
    import pandas as pd
    import numpy as np
    import numpy as np

except Exception as e :
    print("Some Modules are Missing{}".format(e))

In [4]:
test = pd.read_csv("../datasets/testing/final_testing_dataset_with_history_30.csv")
train = pd.read_csv("../datasets/training/final_training_dataset_with_history_30_SMOTENC.csv")
print("testing dataset shape: ",test.shape)
print("training dataset shape : ",train.shape)



testing dataset shape:  (9232, 30)
training dataset shape :  (12342, 30)


In [5]:
dataframe = pd.concat([train,test],axis=0)
dataframe = dataframe.reset_index(drop=True)
print(dataframe.shape)


(21574, 30)


In [155]:
dataframe.to_csv("ai_knowledge_base.csv",index=False)

In [7]:
df = pd.read_csv("../datasets/ai_knowledge_base.csv")
df.shape

(21574, 30)

In [18]:
df

,_source.data.protocol,_source.data.id,_source.rule.firedtimes,_source.rule.mail,_source.rule.level,_source.rule.description,_source.rule.groups,_source.rule.pci_dss,_source.rule.tsc,_source.rule.nist_800_53,...,history._source.data.id.500,T1212,T1068,T1064,T1210,T1083,T1055,T1190,output_2,output_1
0,GET,200,34,0,1,Access log messages grouped.,"[""web"",""accesslog""]",,,,...,1,0,0,0,0,0,0,0,NORMAL,NORMAL
1,GET,200,49,0,1,Access log messages grouped.,"[""web"",""accesslog""]",,,,...,0,0,0,0,0,0,0,0,NORMAL,NORMAL
2,GET,304,3265,0,1,Ignored URLs (simple queries).,"[""web"",""accesslog""]",,,,...,0,0,0,0,0,0,0,0,NORMAL,NORMAL
3,GET,304,1162,0,1,Ignored URLs (simple queries).,"[""web"",""accesslog""]",,,,...,0,0,0,0,0,0,0,0,brute_force,ATTACK
4,GET,500,71,0,6,XSS (Cross Site Scripting) attempt.,"[""web"",""accesslog"",""attack""]","[""6.5"",""11.4"",""6.5.7""]","[""CC6.6"",""CC7.1"",""CC8.1"",""CC6.1"",""CC6.8"",""CC7....","[""SA.11"",""SI.4""]",...,0,0,0,0,0,0,0,1,XSS,ATTACK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21569,GET,500,1009,0,5,Web server 500 error code (Internal Error).,"[""web"",""accesslog"",""system_error""]",,,,...,11,0,0,0,0,4,4,4,WEB_SCAN,ATTACK
21570,GET,200,1860,0,1,Access log messages grouped.,"[""web"",""accesslog""]",,,,...,0,0,0,0,0,0,0,0,SQL_INJECTION,ATTACK
21571,GET,200,2487,0,1,Ignored URLs (simple queries).,"[""web"",""accesslog""]",,,,...,0,0,0,0,0,0,0,3,SENSITIVE_DATA_EXPOSURE,ATTACK
21572,GET,200,665,0,1,Access log messages grouped.,"[""web"",""accesslog""]",,,,...,2,0,0,0,0,1,1,3,SENSITIVE_DATA_EXPOSURE,ATTACK


In [8]:
df[df["output_1"] == " "]

,_source.data.protocol,_source.data.id,_source.rule.firedtimes,_source.rule.mail,_source.rule.level,_source.rule.description,_source.rule.groups,_source.rule.pci_dss,_source.rule.tsc,_source.rule.nist_800_53,...,history._source.data.id.500,T1212,T1068,T1064,T1210,T1083,T1055,T1190,output_2,output_1


In [9]:
def generator(df,index_name):
    
    for index,row in df.iterrows():
        mail = False
        if(row["_source.rule.mail"] == 1):
            mail = True
        yield{
            '_index':index_name,
            '_type':'_doc',
#             '_id':row.get("_id",index),
            '_source':{
                'input_type':row.get("_source.input.type","log"),
                'agent_ip':row.get("_source.agent.ip"," "),
                "agent_name":row.get("_source.agent.ip"," "),
                "agent_id":row.get("_source.agent.id"," "),
                "manager_name":row.get("_source.manager.name"," "),
                'data_protocol':row.get('_source.data.protocol'," "),
                "data_srcip":row.get("_source.data.srcip"," "),
                'data_id':row.get('_source.data.id'," "),
                'data_url':row.get('_source.data.url'," "),
                'rule_firedtimes':row.get('_source.rule.firedtimes',1),
                'rule_mail':mail,
                'rule_level':row.get('_source.rule.level',1),
                'rule_description':row.get("_source.rule.description"," "),
                'rule_groups':row.get("_source.rule.groups"," "),
                'rule_id':row.get("_source.rule.id"," "),
                'location':row.get("_source.location"," "),
                'decoder_name':row.get("_source.decoder.name"," "),
                "id":row.get("_source.id"," "),
                "full_log":row.get("_source.full_log"," "),
                "timestamp":row.get("_source.timestamp","1111-1-11T1:11:1.1111Z"),
                "rule_pci_dss":row.get("_source.rule.pci_dss"," "),
                "rule_tsc":row.get("_source.rule.tsc"," "),
                "rule_nist_800_53":row.get("_source.rule.nist_800_53"," "),
                "rule_gdpr":row.get("_source.rule.gdpr"," "),
                "rule_mitre_id":row.get("_source.rule.mitre.id"," "),
                "rule_hipaa":row.get("_source.rule.hipaa"," "),
                "agent_description":row.get("_source.gent.description"," "),
                "rule_frequency":row.get("_source.rule.frequency"," "),
                "history_rule_firedtimes":row.get('history._source.rule.firedtimes'," "),
                'history_source_data_id_200':row.get('history._source.data.id.200'," "),
                'history_source_data_id_300':row.get('history._source.data.id.300'," "),
                'history_source_data_id_400':row.get('history._source.data.id.400'," "),
                'history_source_data_id_500':row.get('history._source.data.id.500'," "),

                'history_T1212':row.get('T1212'," "),
                'history_T1068':row.get('T1068'," "),
                'history_T1064':row.get('T1064'," "),
                'history_T1210':row.get('T1210'," "),
                'history_T1083':row.get('T1083'," "),
                'history_T1055':row.get('T1055'," "),
                'history_T1190':row.get('T1190'," "),


                "history_count_events":row.get("count_events",0),
                "output_1":row.get("output_1"," "),
                "output_2":row.get("output_2"," ")
                
            }
        }

In [10]:
all_features = ['_source.data.protocol',
                         '_source.data.id',
                         '_source.rule.firedtimes',
                         '_source.rule.mail',
                         '_source.rule.level',
                         '_source.rule.description',
                         '_source.rule.groups',
                         '_source.rule.pci_dss',
                         '_source.rule.tsc',
                         '_source.rule.nist_800_53',
                         '_source.rule.gdpr',
                         '_source.rule.mitre.id',
                         '_source.rule.frequency',
                         '_source.rule.hipaa',
                         '_source.agent.description',
                         '_source.rule.id',
                         'history._source.rule.firedtimes',
                         'history._source.data.id.200',
                         'history._source.data.id.300',
                         'history._source.data.id.400',
                         'history._source.data.id.500',
                         'T1212',
                         'T1068',
                         'T1064',
                         'T1210',
                         'T1083',
                         'T1055',
                         'T1190']
category_features= ['_source.data.protocol',
                         '_source.data.id',
                         '_source.rule.mail',
                         '_source.rule.description',
                         '_source.rule.groups',
                         '_source.rule.pci_dss',
                         '_source.rule.tsc',
                         '_source.rule.nist_800_53',
                         '_source.rule.gdpr',
                         '_source.rule.mitre.id',
                         '_source.rule.hipaa',
                         '_source.agent.description',
                         '_source.rule.frequency',
                         '_source.rule.id']

numerical_features = ['_source.rule.firedtimes',
                      '_source.rule.level',
                      'history._source.rule.firedtimes',
                         'history._source.data.id.200',
                         'history._source.data.id.300',
                         'history._source.data.id.400',
                         'history._source.data.id.500',
                         'T1212',
                         'T1068',
                         'T1064',
                         'T1210',
                         'T1083',
                         'T1055',
                         'T1190']

In [11]:
def preprocessing(df):  
    #these category features if they didn't exist we will add them to the dataframe
    list_test_columns= list(df.columns)
    for i in range ( 0, len(category_features)):
        if(category_features[i] not in list_test_columns):
            df[category_features[i]] = " "
    for c in numerical_features:
        if(c not in list_test_columns):
            df[c] = 1
    # fill the columns that has None values with empty strings.
    for c in category_features:
        if(df[c].isnull().any()):
            df[c] = df[c].fillna(' ')
    # fill the columns rule level with rule.level = 3
    df["_source.rule.level"] = df["_source.rule.level"].fillna(1)
    df["_source.rule.firedtimes"] = df["_source.rule.firedtimes"].fillna(1)

    # fill mail values with 1 and 0 
    for index,row in df.iterrows():

        if(isinstance(row["_source.rule.id"],int)):
            df.at[index,'_source.rule.id'] = str(row['_source.rule.id'])
        
        for cat_fe in category_features:
            if(isinstance(row[cat_fe],list)):
                df.at[index,cat_fe] = str(row[cat_fe])
        if (row["_source.rule.mail"]==True):
            df.at[index,"_source.rule.mail"] = 1
        else :
            df.at[index,"_source.rule.mail"] = 0
    df = df.fillna(' ')
    return df

In [19]:

def write_data_to_elasticsearch(df,es):
    
    index_name ="wazuh-ai-knowledge_base"
    settings={
        "settings":{
            "number_of_shards":1,
            "number_of_replicas":0,     
        },
    "mappings":{
        "properties":{
            "rule.frequency":{
                "type":"text"
            },
            "timestamp":{
                'type':"date",
                "format": "date_optional_time"
            }
        }
    }
    }
    if(not es.indices.exists(index_name)):
        es.indices.create(index=index_name,body=settings)
    print("before bulk",df.shape)
    helpers.bulk(es,generator(df,index_name))
    


In [20]:
def write_to_knowledge_base(es):

    dataframe =pd.read_csv("../datasets/ai_knowledge_base.csv")

    dataframe = preprocessing(dataframe)
   
    print("training dataset shape : ",dataframe.shape)
    write_data_to_elasticsearch(dataframe,es)
    print("Finished!!")
    


In [21]:
def __main__():
    user= "wazuh"
    password = "wazuh"
    elasticsearch_ip="192.168.56.104"
    elasticsearch_port = "9200"
    es = Elasticsearch(["https://" + user + ":"+password+"@"+elasticsearch_ip+":"+elasticsearch_port], verify_certs=False)
    if(es.ping()):
        write_to_knowledge_base(es)
   
    

In [22]:
__main__()

C:\ProgramData\Anaconda3\lib\site-packages\elasticsearch\connection\http_urllib3.py:209: UserWarning: Connecting to https://192.168.56.104:9200 using SSL with verify_certs=False is insecure.
  warnings.warn(
C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.104'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


training dataset shape :  (21574, 30)
before bulk (21574, 30)


C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.104'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\Anaconda3\lib\site-packages\elasticsearch\connection\base.py:208: ElasticsearchWarning: this request accesses system indices: [.kibana_1], but in a future major version, direct access to system indices will be prevented by default
  warnings.warn(message, category=ElasticsearchWarning)
C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.104'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connectionpool.py:104

C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.104'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.104'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.104'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connection

Finished!!
